In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import os
import holidays

# Set your exact local paths
processed_data_path = r'A:\desktop\New folder\P\Ratul\Supply_Chain_Thesis\data\processed\cleaned_data.csv'
models_dir = r'A:\desktop\New folder\P\Ratul\Supply_Chain_Thesis\models'
processed_dir = r'A:\desktop\New folder\P\Ratul\Supply_Chain_Thesis\data\processed'

# Ensure the models directory exists for saving deployment artifacts
os.makedirs(models_dir, exist_ok=True)

print("Loading cleaned dataset...")
df = pd.read_csv(processed_data_path)
df['order_date'] = pd.to_datetime(df['order_date'])
print(f"Data loaded successfully. Shape: {df.shape}")

Loading cleaned dataset...
Data loaded successfully. Shape: (180519, 48)


In [2]:
from sklearn.cluster import MiniBatchKMeans
from textblob import TextBlob
import holidays
import joblib
import os

# 1. Temporal Engineering & Holiday Flags
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_month'] = df['order_date'].dt.month
df['order_day_of_week'] = df['order_date'].dt.dayofweek
df['is_weekend'] = df['order_day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

us_holidays = holidays.US() 
df['is_holiday'] = df['order_date'].dt.date.apply(lambda x: 1 if x in us_holidays else 0)

# 2. Geospatial K-Means Clustering (50 Zones)
kmeans = MiniBatchKMeans(n_clusters=50, random_state=42, n_init=3, batch_size=10000)
df['geospatial_zone'] = kmeans.fit_predict(df[['latitude', 'longitude']])

# 3. Text Metrics: Sentiment & Readability (RP Requirement)
print("Calculating text sentiment and readability...")
df['text_sentiment'] = df['category_name'].fillna('').apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['text_readability_length'] = df['category_name'].fillna('').apply(lambda x: len(str(x).split()))

# 4. Target / Frequency Encoding (RP Requirement)
market_freq = df['market'].value_counts(normalize=True).to_dict()
df['market_freq_encoded'] = df['market'].map(market_freq)

# 5. 7-Day Rolling Averages (RP Requirement)
print("Calculating 7-day rolling averages...")
df = df.sort_values('order_date')
df['zone_7d_rolling_avg'] = df.groupby('geospatial_zone')['days_for_shipping_real'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
).fillna(df['days_for_shipping_real'].mean())

# 6. Capacity Proxies & Delivery Ratios (RP Requirement Section 7.2)
print("Calculating capacity proxies and delivery ratios...")
# Capacity proxy: Sales value per item quantity (since weight isn't available in DataCo)
df['capacity_proxy_value_per_item'] = df['sales'] / df['order_item_quantity'].replace(0, 1)

# Delivery Ratio Proxy: We use the geospatial zone cluster distance proxy
# (Since exact routing distance isn't available, we use the ratio of discount to sales as an economic proxy for shipping priority)
df['economic_delivery_ratio'] = df['order_item_discount'] / df['sales'].replace(0, 1)

# Save the clustering model
os.makedirs('../models', exist_ok=True)
joblib.dump(kmeans, '../models/production_kmeans.pkl')

print("All RP Features (Holidays, 50-Zones, Sentiment, Frequency Encoding, Rolling Averages) computed!")

Calculating text sentiment and readability...
Calculating 7-day rolling averages...
Calculating capacity proxies and delivery ratios...
All RP Features (Holidays, 50-Zones, Sentiment, Frequency Encoding, Rolling Averages) computed!


In [3]:
# Sort chronologically to prevent future data from leaking
df = df.sort_values('order_date').reset_index(drop=True)

# Select predictive features including the FINAL TWO RP features
selected_features = [
    'shipping_mode', 'market', 'customer_segment', 'order_region',
    'geospatial_zone', 'order_month', 'order_day_of_week', 'is_weekend', 'is_holiday',
    'order_item_quantity', 'sales', 'order_item_discount', 'category_name',
    'text_sentiment', 'text_readability_length', 'market_freq_encoded', 'zone_7d_rolling_avg',
    'capacity_proxy_value_per_item', 'economic_delivery_ratio' # <-- The final missing ratios!
]

X = df[selected_features]
y = df['days_for_shipping_real'] 

split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Chronological Split Complete.")

Chronological Split Complete.


In [4]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Identify columns
cat_cols = ['shipping_mode', 'market', 'customer_segment', 'order_region', 'geospatial_zone']

# Add the final two features to num_cols so they get scaled!
num_cols = [
    'order_month', 'order_day_of_week', 'is_weekend', 'is_holiday', 
    'order_item_quantity', 'sales', 'order_item_discount',
    'text_sentiment', 'text_readability_length', 'market_freq_encoded', 'zone_7d_rolling_avg',
    'capacity_proxy_value_per_item', 'economic_delivery_ratio' # <-- Scaled here
]
text_col = 'category_name'

# 1. One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train[cat_cols]), columns=encoder.get_feature_names_out(cat_cols))
X_test_encoded = pd.DataFrame(encoder.transform(X_test[cat_cols]), columns=encoder.get_feature_names_out(cat_cols))

# 2. Standard Scaling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train[num_cols]), columns=num_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test[num_cols]), columns=num_cols)

# 3. TF-IDF Text Engineering
print("Applying TF-IDF to category names...")
tfidf = TfidfVectorizer(max_features=10, stop_words='english')
X_train_text = pd.DataFrame(tfidf.fit_transform(X_train[text_col].fillna('')).toarray(), columns=[f"tfidf_{w}" for w in tfidf.get_feature_names_out()])
X_test_text = pd.DataFrame(tfidf.transform(X_test[text_col].fillna('')).toarray(), columns=[f"tfidf_{w}" for w in tfidf.get_feature_names_out()])

# Combine all Processed Features
X_train_final = pd.concat([X_train_scaled, X_train_encoded, X_train_text], axis=1)
X_test_final = pd.concat([X_test_scaled, X_test_encoded, X_test_text], axis=1)

# Export Artifacts
joblib.dump(encoder, '../models/production_encoder.pkl')
joblib.dump(scaler, '../models/production_scaler.pkl')
joblib.dump(tfidf, '../models/production_tfidf.pkl')
joblib.dump(list(X_train_final.columns), '../models/production_features_list.pkl')

print("Encoding, Scaling, and TF-IDF Text Engineering complete.")

Applying TF-IDF to category names...
Encoding, Scaling, and TF-IDF Text Engineering complete.


In [5]:
# Save the engineered datasets directly to your processed folder
X_train_final.to_csv(os.path.join(processed_dir, 'X_train.csv'), index=False)
X_test_final.to_csv(os.path.join(processed_dir, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(processed_dir, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(processed_dir, 'y_test.csv'), index=False)

print("Engineered datasets saved successfully to data/processed/")

Engineered datasets saved successfully to data/processed/
